# Aula: Introdução às Redes Neurais com PyTorch
**Disciplina:** Tópicos A - Redes Neurais Profundas (2026)
**Prof:** Thiago Medeiros

---

Bem-vindos! Este notebook foi projetado para servir como um material didático e teoricamente denso para introduzir a parte prática e conceitual de Redes Neurais Artificiais (Multi-Layer Perceptrons - MLP) usando o framework PyTorch.

Aqui, você aprenderá as etapas fundamentais necessárias para preparar um conjunto de dados do mundo real, configurar uma arquitetura de rede neural, definir o loop de treinamento, aplicar técnicas de inicialização e regularização, e avaliar os resultados de forma robusta.

### Objetivos do Aprendizado:
1. **Compreensão Teórica de Funções de Ativação**: Entender a necessidade de não-linearidades e os desafios de gradientes em redes profundas.
2. **Ciclo de Vida de Dados Reais**: Tratar variáveis categóricas e numéricas, dividir conjuntos e evitar o vazamento de dados (*Data Leakage*).
3. **Mecânica do Treinamento no PyTorch**: Entender o funcionamento do Autograd, o grafo de computação dinâmico e o loop clássico de otimização.
4. **Regularização e Inicialização de Pesos**: Analisar na prática o efeito de inicializações como He (Kaiming), bem como Dropout, Batch Normalization e Regularização L2.
5. **Desafio Autônomo**: Implementar um pipeline de processamento para um novo dataset (Penguins) e estender o loop de treino com decaimento da taxa de aprendizado (*Learning Rate Decay*).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Configurar sementes para garantir reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

# Configurar dispositivo de execução (GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo de execução atual: {device}")

## 1. Fundamentos Teóricos e Funções de Ativação

### Do Perceptron à MLP
O Perceptron original calcula uma combinação linear de suas entradas com pesos ($w$) e viés ($b$):
$$z = \sum_{i=1}^{n} w_i x_i + b = w^T x + b$$

Se o resultado $z$ for maior que um certo limiar, o neurônio dispara (saída 1), caso contrário, não dispara (saída 0). Essa formulação clássica é estritamente linear, o que limita o Perceptron a resolver apenas problemas linearmente separáveis (por exemplo, ele não consegue resolver o clássico operador XOR).

Para aprender padrões complexos, combinamos múltiplos neurônios organizados em camadas: uma camada de entrada, uma ou mais camadas ocultas (*hidden layers*), e uma camada de saída. Essa arquitetura é conhecida como **Multi-Layer Perceptron (MLP)** ou rede *feedforward*.

### Por que precisamos de funções de ativação não-lineares?
Pelo **Teorema da Aproximação Universal**, uma rede neural *feedforward* com uma única camada oculta contendo um número finito de neurônios pode aproximar qualquer função contínua em subconjuntos compactos do $\mathbb{R}^n$, desde que utilize funções de ativação não-lineares.

Caso usássemos funções puramente lineares, o empilhamento de camadas seria inútil. Matematicamente, a composição de múltiplas transformações lineares é equivalente a uma única transformação linear:
$$y = W_2(W_1 x + b_1) + b_2 = (W_2 W_1)x + (W_2 b_1 + b_2) = W' x + b'$$
Portanto, sem a não-linearidade, a rede seria incapaz de aprender limites de decisão complexos e curvas não-lineares.

### Funções de Ativação Mais Utilizadas

#### 1. Sigmoid (ou Logística)
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
- **Comportamento:** Achata os valores de entrada para o intervalo $(0, 1)$, ideal para interpretar saídas como probabilidades na classificação binária.
- **Limitação (Vanishing Gradient):** Suas derivadas nas extremidades são próximas de zero. Quando a ativação $z$ assume valores muito grandes ou muito pequenos, o gradiente $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ tende a zero. Durante a retropropagação (*backpropagation*), esse gradiente nulo é multiplicado em cadeia, fazendo com que as camadas iniciais parem de atualizar seus pesos (a rede "para" de aprender).

#### 2. Tangente Hiperbólica (tanh)
$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$
- **Comportamento:** Mapeia a entrada para o intervalo $(-1, 1)$. Possui a vantagem de ser centrada no zero (média das saídas próxima a 0), o que auxilia na aceleração do aprendizado durante a otimização.
- **Limitação:** Assim como a Sigmoid, sofre com saturação e desvanecimento do gradiente nas extremidades.

#### 3. ReLU (Rectified Linear Unit)
$$\text{ReLU}(z) = \max(0, z)$$
- **Comportamento:** Retorna $z$ se a entrada for positiva e 0 caso contrário.
- **Vantagens:** É extremamente eficiente para calcular (não usa exponenciais) e resolve o problema do vanishing gradient no domínio positivo, já que sua derivada é constante e igual a 1 para $z > 0$.
- **Limitação (Dying ReLU):** Se um neurônio recebe entradas que geram saídas negativas de forma consistente durante o treino, seu gradiente será exatamente 0. Esse neurônio entra em um estado inativo definitivo ("morre"), deixando de contribuir para o aprendizado.

#### 4. Softmax
$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}$$
- **Finalidade:** Utilizada na última camada para classificação multiclasse ($C > 2$). Ela converte os valores brutos da rede (*logits*) em uma distribuição probabilística normalizada sobre as classes existentes (a soma de todas as probabilidades de saída será exatamente igual a 1).

## 2. Ciclo de Vida dos Dados e Prevenção de Data Leakage

O sucesso no treinamento de uma rede neural começa na preparação rigorosa dos dados. Abordaremos dois pontos fundamentais:

### Divisão dos Dados: Treino, Validação e Teste
1. **Treino (Train):** Usado pelo algoritmo para ajustar os pesos e bias do modelo.
2. **Validação (Validation):** Usado para avaliar o modelo durante o treinamento e guiar decisões sobre hiperparâmetros (como taxa de aprendizado e regularizações). Ele nos diz se a rede está sofrendo *overfitting*.
3. **Teste (Test):** Conjunto mantido oculto durante todo o desenvolvimento. Só é acessado uma única vez após a conclusão de todo o processo de tuning para fornecer uma estimativa realista do poder de generalização do modelo.

### O Perigo do Data Leakage (Vazamento de Dados)
O *data leakage* ocorre quando informações fora do conjunto de treinamento vazam para dentro do processo de treino. Um exemplo clássico e frequente é a normalização de variáveis numéricas. 

Para normalizar as colunas (padronizar para média = 0 e desvio padrão = 1), precisamos calcular a média ($\mu$) e o desvio padrão ($\sigma$). Se calcularmos estas estatísticas considerando **todo o conjunto de dados** antes de fazer a divisão de treino/validação/teste, o conjunto de treino receberá influência das médias e variações dos conjuntos de validação e teste. Isso gera uma estimativa excessivamente otimista do modelo durante o treino e pode causar sérias falhas em produção.

**A regra de ouro:** Ajuste (`fit`) o pré-processador (como o `StandardScaler`) **apenas no conjunto de treino**, e então aplique o pré-processador ajustado (`transform`) nos conjuntos de treino, validação e teste.

Para demonstrar essas boas práticas, utilizaremos o dataset **Titanic**, que contém variáveis numéricas e categóricas misturadas.

In [ ]:
# Carregar o dataset real do Titanic pelo Seaborn
titanic = sns.load_dataset('titanic')
print(f"Formato original: {titanic.shape}")
titanic.head()

### Limpeza e Seleção de Atributos
Para nossa rede, selecionaremos um conjunto de atributos misto:
- **Numéricos:** `age`, `fare`, `sibsp` (nº de irmãos/cônjuge a bordo), `parch` (nº de pais/filhos a bordo).
- **Categóricos:** `sex` (gênero), `embarked` (porto de embarque: C, Q, S), `pclass` (classe do bilhete: 1, 2, 3).
- **Alvo (Target):** `survived` (0 para óbito, 1 para sobrevivente).

In [ ]:
# Selecionar colunas de interesse
cols_features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target_col = 'survived'
df = titanic[cols_features + [target_col]].copy()

# 1. Tratamento de Valores Ausentes (Imputação)
# Para didática simples, preenchemos idades ausentes com a mediana e porto com a moda
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Verificar se ainda existem valores nulos
print("Valores ausentes tratados:\n", df.isnull().sum())

### Codificação de Variáveis Categóricas (One-Hot Encoding)
Redes neurais não conseguem processar strings de texto diretamente. É necessário convertê-las em números. A técnica de **One-Hot Encoding** cria uma nova coluna binária (0 ou 1) para cada categoria exclusiva de uma variável.

No Pandas, usamos a função `pd.get_dummies()`. Usamos o parâmetro `drop_first=True` para evitar a colinearidade (a chamada armadilha das dummies), onde a informação de uma coluna pode ser perfeitamente predita pelas outras (por exemplo, se `sex_male` é 0, implicitamente sabemos que a pessoa pertence ao gênero feminino, tornando a coluna `sex_female` redundante).

In [ ]:
# Definir 'pclass' como string para que o pandas a trate como categórica e gere dummies
df['pclass'] = df['pclass'].astype(str)

# Aplicar One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['pclass', 'sex', 'embarked'], drop_first=True)
print(f"Novas colunas criadas:\n{df_encoded.columns.tolist()}")
df_encoded.head()

### Divisão dos Dados e Padronização (Evitando o Vazamento de Dados)
Agora faremos a separação das variáveis independentes (X) e dependente (y) e a partição do dataset. Em seguida, normalizaremos as variáveis contínuas aplicando o `StandardScaler` de forma correta.

In [ ]:
# Separar features e target
X_data = df_encoded.drop(columns=[target_col]).astype(np.float32)
y_data = df_encoded[target_col].values.astype(np.float32)

# Dividir primeiro em Treino (70%) e Temporário (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, test_size=0.30, random_state=42, stratify=y_data
)

# Dividir o Temporário ao meio para gerar Validação (15%) e Teste (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Padronização numérica sem Data Leakage
num_cols = ['age', 'fare', 'sibsp', 'parch']

scaler = StandardScaler()

# Ajustar o scaler APENAS na partição de Treino
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# Aplicar o scaler ajustado nas partições de Validação e Teste
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Tamanho das divisões: Treino ({X_train.shape[0]}), Val ({X_val.shape[0]}), Teste ({X_test.shape[0]})")

### Encapsulamento em Datasets e DataLoaders do PyTorch
A infraestrutura de carregamento de dados do PyTorch depende de duas classes fundamentais:
- `TensorDataset`: Agrupa os tensores de entrada e saída por amostra.
- `DataLoader`: Gerencia a iteração sobre os dados. Ele é responsável pelo fatiamento dos batches, embaralhamento (`shuffle`) e paralelização no carregamento.

**Boas práticas de Shuffling:** Ative `shuffle=True` apenas para os dados de **Treino**. Isso força a rede a ver os dados em ordens diferentes a cada época, prevenindo que ela aprenda a sequência das amostras. Para validação e teste, mantenha `shuffle=False` para garantir consistência e estabilidade na avaliação.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 1. Converter DataFrames/Arrays NumPy para Tensores do PyTorch
train_ds = TensorDataset(torch.tensor(X_train.values), torch.tensor(y_train))
val_ds   = TensorDataset(torch.tensor(X_val.values), torch.tensor(y_val))
test_ds  = TensorDataset(torch.tensor(X_test.values), torch.tensor(y_test))

# 2. Instanciar DataLoaders (Tamanho do Batch = 32)
BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Exemplo de amostragem de lote para inspeção
x_sample, y_sample = next(iter(train_loader))
print(f"Formato das features no lote: {x_sample.shape}")
print(f"Formato dos targets no lote:   {y_sample.shape}")

## 3. Construindo a Arquitetura da MLP no PyTorch

No PyTorch, construímos uma classe herdando de `nn.Module`. Definimos os blocos de construção da rede (as camadas) no método inicializador `__init__`, e o fluxo de passagem dos dados no método `forward`.

Criaremos uma arquitetura básica inicial chamada `TitanicMLP` contendo:
- Uma camada oculta linear com 32 neurônios + ativação ReLU.
- Uma segunda camada oculta linear com 16 neurônios + ativação ReLU.
- Uma camada de saída linear de dimensão 1.

*Nota Didática sobre a Saída:* A camada de saída retorna apenas um número real chamado de **logit**. Não aplicamos a função Sigmoid diretamente no final da nossa rede porque usaremos a perda `nn.BCEWithLogitsLoss()`. Essa função de perda recebe logits brutos e realiza a operação matemática combinada de Sigmoid + Binary Cross Entropy de forma otimizada e numericamente estável, evitando problemas de Underflow/Overflow sob o ponto flutuante.

In [ ]:
class TitanicMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Camadas Lineares (Fully Connected)
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)  # 1 neurônio de saída (classificação binária)
        
        # Função de ativação
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Conectar a passagem de dados
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # Retorna o logit bruto
        return x.squeeze()  # Remove dimensões extras (ex: [32, 1] vira [32])

# Instanciar e enviar o modelo para o dispositivo correto (CPU ou GPU)
input_dimension = X_train.shape[1]
model_basic = TitanicMLP(input_dim=input_dimension).to(device)
print(model_basic)

## 4. O Grafo de Computação e o Loop de Treinamento

O ciclo de treinamento de uma rede neural segue um fluxo iterativo rígido que repete os passos de **Forward**, cálculo da perda (**Loss**), **Backward** e atualização dos pesos (**Optimizer Step**).

### Anatomia do Loop de Treinamento no PyTorch
A cada batch de dados:
1. **`optimizer.zero_grad()`:** Os gradientes dos tensores no PyTorch são acumulados por padrão para facilitar o design de arquiteturas complexas (como RNNs). Por isso, no início de cada batch, devemos zerá-los manualmente.
2. **`outputs = model(X_batch)`:** Executa o passo *forward*, passando os dados pelas camadas do modelo e calculando os logits de predição.
3. **`loss = criterion(outputs, y_batch)`:** Calcula o erro do modelo utilizando a função de perda (Cross Entropy).
4. **`loss.backward()`:** O motor Autograd realiza a retropropagação. Ele calcula a derivada parcial da perda em relação a cada peso/viés ajustável da rede que possui `requires_grad=True` usando a Regra da Cadeia.
5. **`optimizer.step()`:** O otimizador utiliza os gradientes recém-calculados para atualizar os pesos, movendo-os na direção oposta ao gradiente (para minimizar a perda).

Vamos implementar uma função de treino robusta que computa o histórico de perda e acurácia tanto para o conjunto de **Treino** quanto para o de **Validação**.

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100):
    # Dicionário para armazenar o histórico de métricas
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }
    
    for epoch in range(epochs):
        # --- PASSO 1: FASE DE TREINAMENTO ---
        model.train()  # Habilita comportamentos como Dropout/BatchNorm
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for X_batch, y_batch in train_loader:
            # Enviar dados ao dispositivo correto (GPU ou CPU)
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # Zerar gradientes acumulados
            optimizer.zero_grad()
            
            # Forward
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            # Backward
            loss.backward()
            
            # Otimização
            optimizer.step()
            
            # Métricas
            running_loss += loss.item() * X_batch.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct_train += (preds == y_batch).sum().item()
            total_train += y_batch.size(0)
            
        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = correct_train / total_train
        
        # --- PASSO 2: FASE DE VALIDAÇÃO ---
        model.eval()   # Desabilita Dropout/BatchNorm para inferência consistente
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        # Desligar o rastreamento de gradientes para economizar memória e desempenho
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                running_val_loss += loss.item() * X_batch.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                correct_val += (preds == y_batch).sum().item()
                total_val += y_batch.size(0)
                
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc = correct_val / total_val
        
        # Armazenar histórico
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        # Log a cada 10 épocas
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Época {epoch+1:03d}/{epochs} | "
                  f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc*100:.2f}% | "
                  f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc*100:.2f}%")
            
    return history

In [ ]:
# Inicializar o critério de perda e o otimizador
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_basic.parameters(), lr=0.005)

# Iniciar o treinamento por 80 épocas
print("Treinando o modelo básico...")
history_basic = train_model(model_basic, train_loader, val_loader, criterion, optimizer, epochs=80)

In [ ]:
# Plotar o histórico de treinamento
plt.figure(figsize=(14, 5))

# Gráfico de Perda
plt.subplot(1, 2, 1)
plt.plot(history_basic['train_loss'], label='Loss - Treino', color='#1f77b4')
plt.plot(history_basic['val_loss'], label='Loss - Validação', color='#ff7f0e', linestyle='--')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.title('Histórico de Perda (Loss)')
plt.legend()

# Gráfico de Acurácia
plt.subplot(1, 2, 2)
plt.plot(history_basic['train_acc'], label='Acc - Treino', color='#2ca02c')
plt.plot(history_basic['val_acc'], label='Acc - Validação', color='#d62728', linestyle='--')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.title('Histórico de Acurácia')
plt.legend()

plt.tight_layout()
plt.show()

## 5. Inicialização de Pesos e Técnicas de Regularização

Ao analisarmos os gráficos anteriores, é comum notar sintomas de **Overfitting** (quando o modelo decora o treino: a curva de perda de treino cai continuamente, mas a perda de validação estabiliza ou começa a subir). Vamos introduzir técnicas para mitigar esse comportamento e garantir estabilidade numérica ao modelo.

### 5.1 A Importância da Inicialização de Pesos
Por padrão, o PyTorch inicializa as camadas lineares com valores pequenos baseados na dimensão de entrada da camada. Se definirmos manualmente ou não tomarmos cuidado com a inicialização, podemos arruinar o aprendizado:
- **Todos os pesos em zero:** Todos os neurônios da mesma camada oculta calcularão a mesma ativação e receberão os mesmos gradientes. A rede perderá a capacidade de quebrar a simetria, agindo como se tivesse apenas um único neurônio por camada.
- **Pesos excessivamente grandes:** Causam saturação de ativações saturantes (Sigmoid/Tanh), provocando vanishing gradient imediato.

#### Inicialização He (Kaiming)
Idealizada especificamente para camadas que utilizam ativações do tipo **ReLU**. Ela compensa o fato de que a ReLU colapsa metade da distribuição para zero. Ela define que os pesos devem ser amostrados de uma distribuição normal com variância calculada como:
$$\text{Var}(W) = \frac{2}{n_{in}}$$
Onde $n_{in}$ é o número de conexões de entrada do neurônio.

### 5.2 Técnicas de Regularização

#### 1. Regularização L2 (Weight Decay)
Adiciona um termo de penalização à perda padrão que é diretamente proporcional à magnitude quadrada dos pesos do modelo:
$$L_{regularizada} = L_{original} + \lambda \sum_{w} w^2$$
Isso força os pesos a permanecerem pequenos, restringindo a complexidade do modelo e suavizando a fronteira de decisão. No PyTorch, isso é implementado de forma transparente passando o argumento `weight_decay` ($\lambda$) ao otimizador.

#### 2. Dropout
Durante a fase de treinamento, o Dropout desativa de forma aleatória e independente uma fração $p$ (ex: 30%) das ativações de saída de uma camada. Isso força a rede a aprender representações distribuídas e redundantes, impedindo que ela confie demais em caminhos ou neurônios específicos.

**Atenção:** Durante a avaliação (`model.eval()`), o Dropout é desativado automaticamente, e as saídas são escaladas proporcionalmente pelo fator $(1-p)$ para compensar.

#### 3. Batch Normalization (Normalização em Lotes)
Consiste em normalizar os dados intermediários que passam pelas camadas internas da rede durante o próprio processamento do batch. Ela calcula a média e a variância de cada *feature* dentro do mini-batch atual, centraliza e reescala os valores. Isso reduz o que chamamos de *Internal Covariate Shift* (mudança na distribuição das ativações conforme os pesos das camadas anteriores se movem).

**Atenção:** Assim como o Dropout, o Batch Norm se comporta diferente em treino (calcula médias/variâncias do batch) vs avaliação (utiliza estatísticas históricas acumuladas).

In [ ]:
class TitanicMLPRegularized(nn.Module):
    def __init__(self, input_dim, dropout_prob=0.3):
        super().__init__()
        
        # Camada 1 + Batch Normalization + ReLU + Dropout
        self.fc1 = nn.Linear(input_dim, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=dropout_prob)
        
        # Camada 2 + Batch Normalization + ReLU + Dropout
        self.fc2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=dropout_prob)
        
        # Camada de Saída (Logit)
        self.fc3 = nn.Linear(32, 1)
        
    def forward(self, x):
        # Conectar as camadas com normalizações e dropouts intercalados
        x = self.dropout1(self.relu1(self.bn1(self.fc1(x))))
        x = self.dropout2(self.relu2(self.bn2(self.fc2(x))))
        x = self.fc3(x)
        return x.squeeze()

In [ ]:
# Função personalizada para inicializar os pesos com o método Kaiming (He) Normal
def init_weights(m):
    if isinstance(m, nn.Linear):
        # Inicialização He para os pesos das camadas lineares
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            # Inicializar viés (bias) com zero
            nn.init.constant_(m.bias, 0.0)

# Instanciar o modelo regularizado
model_reg = TitanicMLPRegularized(input_dim=input_dimension).to(device)

# Aplicar a inicialização de pesos recursivamente a todas as camadas do modelo
model_reg.apply(init_weights)

# Definir critério e otimizador com regularização L2 (Weight Decay = 1e-4)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(
    model_reg.parameters(), 
    lr=0.005, 
    weight_decay=1e-4  # L2 Regularization
)

print("Treinando o modelo regularizado...")
history_reg = train_model(model_reg, train_loader, val_loader, criterion, optimizer, epochs=80)

In [ ]:
# Plotar comparação da perda de validação
plt.figure(figsize=(10, 6))
plt.plot(history_basic['val_loss'], label='Val Loss - Modelo Básico', color='#ff7f0e', linestyle='--')
plt.plot(history_reg['val_loss'], label='Val Loss - Modelo Regularizado + He Init', color='#2ca02c')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.title('Comparativo de Loss na Validação')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 5.3 Avaliação Final do Modelo no Conjunto de Teste

Utilizaremos as amostras de **Teste** (nunca vistas no treinamento) para calcular métricas padrão e gerar uma **Matriz de Confusão**.

Métricas calculadas:
- **Acurácia:** Proporção total de acertos.
- **Precisão:** Das pessoas que a rede predisse que sobreviveram, quantas de fato sobreviveram.
- **Recall (Revogação):** De todos os sobreviventes reais, quantos a rede conseguiu detectar.
- **F1-Score:** Média harmônica entre precisão e recall (indicado para classes desbalanceadas).

In [ ]:
# Colocar o modelo em modo de avaliação
model_reg.eval()

y_true_list = []
y_pred_list = []

# Garantir que gradientes não sejam calculados
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model_reg(X_batch)
        
        # Aplicar Sigmoid para converter logits em probabilidades
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float().cpu().numpy()
        
        y_true_list.extend(y_batch.numpy())
        y_pred_list.extend(preds)

# Calcular métricas
acc = accuracy_score(y_true_list, y_pred_list)
prec = precision_score(y_true_list, y_pred_list)
rec = recall_score(y_true_list, y_pred_list)
f1 = f1_score(y_true_list, y_pred_list)

print("================ METRICAS NO CONJUNTO DE TESTE ================")
print(f"Acurácia:  {acc * 100:.2f}%")
print(f"Precisão:  {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("===============================================================")

# Matriz de Confusão
cm = confusion_matrix(y_true_list, y_pred_list)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Óbito', 'Sobreviveu'], 
            yticklabels=['Óbito', 'Sobreviveu'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão - Modelo Regularizado (Teste)')
plt.show()

## 6. Desafio Prático para os Alunos (Mãos à Obra!)

Agora é a sua vez de aplicar os conceitos discutidos neste laboratório. Siga atentamente as instruções abaixo para resolver as duas partes do desafio.

### Parte 1: Construção de Pipeline para o Dataset `Penguins`
Você deve construir um pipeline de pré-processamento completo e carregar os dados para treinar um modelo de **Classificação Multiclasse**.

1. Carregue o conjunto de dados `Penguins` utilizando o Seaborn: `sns.load_dataset('penguins')`.
2. Limpe registros que possuam qualquer valor ausente (`NaN`) utilizando `.dropna()`.
3. Defina a variável alvo `species` (Espécie do pinguim: Adelie, Chinstrap ou Gentoo). Como este é um problema **multiclasse** (3 classes), mapeie o rótulo de texto em formato de string para inteiros ordinais ($0, 1, 2$).
4. Converta as variáveis categóricas de entrada (`island` e `sex`) em colunas numéricas usando o One-Hot Encoding (`pd.get_dummies` com `drop_first=True`).
5. Divida as features e o alvo em **Treino (70%)**, **Validação (15%)** e **Teste (15%)** usando `train_test_split()`, mantendo a estratificação do target.
6. Identifique as colunas numéricas de entrada: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` e `body_mass_g`. Padronize-as usando `StandardScaler` **sem causar vazamento de dados (Data Leakage)**.
7. Converta os arrays finais em tensores do PyTorch, crie instâncias de `TensorDataset` e configure os `DataLoader` de forma apropriada (com tamanho de batch = 16). Lembre-se das boas práticas de embaralhamento (`shuffle`).

### Parte 2: Modificação do Loop de Treinamento com Learning Rate Scheduler
Taxas de aprendizado constantes nem sempre são ideais: no início do treino, uma taxa alta ajuda a rede a sair rapidamente de mínimos locais; mais tarde, reduzir essa taxa ajuda os pesos a convergirem de maneira suave para o mínimo global.

1. Crie uma nova função de treinamento chamada `train_model_with_scheduler` baseada na função de treino anterior.
2. Essa nova função deve aceitar um parâmetro extra `scheduler` (como por exemplo o `torch.optim.lr_scheduler.StepLR` do PyTorch).
3. No final de cada época (fora do loop de batches), execute o passo do agendador: `scheduler.step()`.
4. Altere a mensagem de log impressa a cada 10 épocas para incluir a taxa de aprendizado atual da época. Você pode recuperá-la acessando `optimizer.param_groups[0]['lr']`.
5. Implemente uma MLP multiclasse simples no PyTorch (a última camada linear deve possuir 3 saídas brutas - logits). A ativação e perda adequada será `nn.CrossEntropyLoss` (que combina Softmax e CrossEntropy).
6. Inicialize os pesos do modelo usando o método He/Kaiming.
7. Configure um otimizador Adam, crie o Scheduler `StepLR` (por exemplo, com `step_size=20` e `gamma=0.5`), e execute o treinamento.
8. Avalie a acurácia final no conjunto de testes de Penguins.

In [ ]:
# ======================================================================
# ESPAÇO DO ALUNO - PARTE 1 (Pipeline de Dados do Dataset Penguins)
# ======================================================================

# 1. Carregar e limpar o dataset
# penguins = sns.load_dataset('penguins')
# penguins = penguins.dropna()

# 2. Mapear o target 'species' para valores ordinais (0, 1, 2)

# 3. Codificar variáveis categóricas de entrada ('island', 'sex')

# 4. Dividir em Treino, Validação e Teste (70%, 15%, 15%)

# 5. Padronizar variáveis numéricas sem data leakage

# 6. Criar Tensores, TensorDataset e DataLoader (Batch Size = 16)

In [ ]:
# ======================================================================
# ESPAÇO DO ALUNO - PARTE 2 (MLP Multiclasse e Loop com LR Scheduler)
# ======================================================================

# 1. Implementar a função train_model_with_scheduler

# 2. Definir a classe PenguinsMLP (Saída de dimensão 3)

# 3. Instanciar o modelo, inicializar os pesos (Kaiming), definir critério (CrossEntropyLoss)
#    otimizador (Adam ou SGD) e o scheduler (ex: StepLR)

# 4. Executar o treinamento por 60 épocas

# 5. Avaliar o modelo treinado no conjunto de testes (imprimir a acurácia)